## Import libraries

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client
from neo4j import GraphDatabase

## Load environment variables

In [4]:
# Load environment variables and retrieve credentials from the .env file
load_dotenv(override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

## Connect to Supabase and Neo4j Aura

In [24]:
# Connect to Supabase and Neo4j Aura
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Successfully connected to Supabase!")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Successfully connected to Neo4j Aura!")

Successfully connected to Supabase!
Successfully connected to Neo4j Aura!


## Data Extraction

In [6]:
# Fetch records from a Supabase table in batches
def fetch_all_rows(table_name, batch_size=1000, limit=None):
    all_rows = []
    start = 0

    while True:
        end = start + batch_size - 1
        if limit is not None:
            end = min(end, limit - 1)

        response = (
            supabase.table(table_name).select("*").order("transaction_id" if table_name == "transactions" else "customer_id")
            .range(start, end).execute())

        rows = response.data
        if not rows:
            break

        all_rows.extend(rows)
        start += len(rows)
        if len(rows) < batch_size:
            break

        if limit is not None and len(all_rows) >= limit:
            break
    return all_rows

In [7]:
transactions_raw = fetch_all_rows(table_name="transactions", batch_size=1000, limit=5000)
transactions_df = pd.DataFrame(transactions_raw)
print("Transactions loaded:", len(transactions_df))
transactions_df.head()

Transactions loaded: 5000


,transaction_id,customer_id,merchant_id,device_id,amount_paise,transaction_amount,payment_channel,merchant_category,transaction_time,customer_age,customer_location,merchant_location,account_age,previous_transactions,average_transaction_value,failed_transactions,split,is_synthetic
0,TX_0000000001,C_b90ff711ce3e3dba,M_11944cb1c387facc,D_e3a0dd890f209d38,79038,790.38,UPI,gift_cards,2024-09-30T18:35:15.602+00:00,52,None,"{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}",836,0,NaN,0,warmup,True
1,TX_0000000002,C_b90ff711ce3e3dba,M_11944cb1c387facc,D_e3a0dd890f209d38,231980,2319.80,UPI,gift_cards,2024-09-30T18:36:10.733+00:00,52,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}",836,1,790.38,1,warmup,True
2,TX_0000000003,C_b90ff711ce3e3dba,M_11944cb1c387facc,D_e3a0dd890f209d38,113282,1132.82,UPI,gift_cards,2024-09-30T18:37:47.985+00:00,52,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}",836,2,1555.09,2,warmup,True
3,TX_0000000004,C_b90ff711ce3e3dba,M_11944cb1c387facc,D_e3a0dd890f209d38,32520,325.20,UPI,gift_cards,2024-09-30T18:38:19.318+00:00,52,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}",836,3,1414.33,2,warmup,True
4,TX_0000000005,C_b90ff711ce3e3dba,M_11944cb1c387facc,D_e3a0dd890f209d38,45515,455.15,UPI,gift_cards,2024-09-30T18:39:15.746+00:00,52,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}",836,4,1142.05,3,warmup,True


In [8]:
# Fetch records from a Supabase table that match the given transaction IDs 
def fetch_by_ids(table_name, column_name, ids, batch_size=500):
    all_rows = []
    unique_ids = list(set(value for value in ids if value is not None))
    for i in range(0, len(unique_ids), batch_size):
        batch_ids = unique_ids[i:i + batch_size]
        response = (
            supabase
            .table(table_name)
            .select("*")
            .in_(column_name, batch_ids)
            .execute()
        )
        all_rows.extend(response.data)
    return all_rows

In [9]:
customers_raw = fetch_by_ids("customers", "customer_id", transactions_df["customer_id"])
customers_df = pd.DataFrame(customers_raw)

devices_raw = fetch_by_ids("devices", "device_id", transactions_df["device_id"])
devices_df = pd.DataFrame(devices_raw)

merchants_raw = fetch_by_ids("merchants","merchant_id", transactions_df["merchant_id"])
merchants_df = pd.DataFrame(merchants_raw)

labels_raw = fetch_by_ids("labels", "transaction_id", transactions_df["transaction_id"])
labels_df = pd.DataFrame(labels_raw)

In [10]:
print("CUSTOMERS:", len(customers_df))
print("DEVICES:", len(devices_df))
print("MERCHANTS:", len(merchants_df))
print("TRANSACTIONS:", len(transactions_df))
print("LABELS:", len(labels_df))

CUSTOMERS: 1577
DEVICES: 1705
MERCHANTS: 200
TRANSACTIONS: 5000
LABELS: 5000


## Data Preparation

- Customer-Device relationships (USED_DEVICE)
- Customer-Merchant relationships (PAID)

In [11]:
print("Transaction columns:")
print(transactions_df.columns.tolist())

print("\nLabel columns:")
print(labels_df.columns.tolist())

Transaction columns:
['transaction_id', 'customer_id', 'merchant_id', 'device_id', 'amount_paise', 'transaction_amount', 'payment_channel', 'merchant_category', 'transaction_time', 'customer_age', 'customer_location', 'merchant_location', 'account_age', 'previous_transactions', 'average_transaction_value', 'failed_transactions', 'split', 'is_synthetic']

Label columns:
['transaction_id', 'fraud_label', 'label_source', 'label_available_at', 'split', 'eligible_at_next_split_start', 'is_synthetic']


In [12]:
transactions_df = transactions_df.merge(labels_df[["transaction_id", "fraud_label"]], on="transaction_id", how="left")
transactions_df[["transaction_id", "fraud_label"]].head()

,transaction_id,fraud_label
0,TX_0000000001,0
1,TX_0000000002,0
2,TX_0000000003,0
3,TX_0000000004,0
4,TX_0000000005,0


In [13]:
print("Duplicate transaction IDs:",
      transactions_df["transaction_id"].duplicated().sum())

print("\nMissing values:")
print(transactions_df.isnull().sum())

Duplicate transaction IDs: 0

Missing values:
transaction_id                  0
customer_id                     0
merchant_id                     0
device_id                      87
amount_paise                    0
transaction_amount              0
payment_channel                 0
merchant_category               0
transaction_time                0
customer_age                    0
customer_location             136
merchant_location               0
account_age                     0
previous_transactions           0
average_transaction_value    1577
failed_transactions             0
split                           0
is_synthetic                    0
fraud_label                     0
dtype: int64


In [14]:
print("Customers:", customers_df["customer_id"].nunique())
print("Devices:", devices_df["device_id"].nunique())
print("Merchants:", merchants_df["merchant_id"].nunique())

print("\nUnique transaction IDs:", transactions_df["transaction_id"].nunique())
print("Unique customers in transactions:", transactions_df["customer_id"].nunique())
print("Unique devices in transactions:", transactions_df["device_id"].nunique())
print("Unique merchants in transactions:", transactions_df["merchant_id"].nunique())

Customers: 1577
Devices: 1705
Merchants: 200

Unique transaction IDs: 5000
Unique customers in transactions: 1577
Unique devices in transactions: 1705
Unique merchants in transactions: 200


In [15]:
print(transactions_df["fraud_label"].value_counts(dropna=False))

fraud_label
0    4975
1      25
Name: count, dtype: int64


In [16]:
# Convert transaction timestamps to UTC datetime format and check invalid values and the overall time range
transactions_df["transaction_time_parsed"] = pd.to_datetime(transactions_df["transaction_time"], errors="coerce", utc=True)
print("\nInvalid timestamps:", transactions_df["transaction_time_parsed"].isna().sum())

print("\nTransaction time range:")
print("Minimum:", transactions_df["transaction_time_parsed"].min())
print("Maximum:", transactions_df["transaction_time_parsed"].max())


Invalid timestamps: 7

Transaction time range:
Minimum: 2024-09-30 18:35:15.602000+00:00
Maximum: 2024-11-15 13:46:28.423000+00:00


In [18]:
# Replace missing device IDs with None
neo4j_df = transactions_df.copy()
neo4j_df["device_id"] = neo4j_df["device_id"].where(neo4j_df["device_id"].notna(), None)

In [20]:
# Prepare transaction records in a dictionary format suitable for inserting into Neo4j
transactions_for_neo4j = []

for row in neo4j_df.to_dict("records"):
    transaction = {
        "transaction_id": row["transaction_id"],
        "customer_id": row["customer_id"],
        "device_id": row["device_id"],
        "merchant_id": row["merchant_id"],
        "amount": float(row["transaction_amount"]),
        "time": (
            row["transaction_time_parsed"].to_pydatetime()
            if pd.notna(row["transaction_time_parsed"])
            else None
        ),
        "fraud_label": int(row["fraud_label"])
    }
    transactions_for_neo4j.append(transaction)

In [21]:
print("Total prepared transactions:", len(transactions_for_neo4j))
print("Transactions with missing device IDs:", sum(t["device_id"] is None for t in transactions_for_neo4j))
print("Transactions with invalid timestamps:", sum(t["time"] is None for t in transactions_for_neo4j))
print("\nSample prepared record:")
print(transactions_for_neo4j[0])

Total prepared transactions: 5000
Transactions with missing device IDs: 0
Transactions with invalid timestamps: 7

Sample prepared record:
{'transaction_id': 'TX_0000000001', 'customer_id': 'C_b90ff711ce3e3dba', 'device_id': 'D_e3a0dd890f209d38', 'merchant_id': 'M_11944cb1c387facc', 'amount': 790.38, 'time': datetime.datetime(2024, 9, 30, 18, 35, 15, 602000, tzinfo=datetime.timezone.utc), 'fraud_label': 0}


## Create the constraints

- A customer may make hundreds of transactions, but should appear as only one Customer node.
- A device may be used by multiple customers, but should appear as only one Device node.
- A merchant may receive thousands of payments, but should appear as only one Merchant node.

In [25]:
# Create uniqueness constraints to prevent duplicate Customer, Device and Merchant nodes 
constraints = [
    """
    CREATE CONSTRAINT customer_id_unique IF NOT EXISTS
    FOR (c:Customer)
    REQUIRE c.customer_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT device_id_unique IF NOT EXISTS
    FOR (d:Device)
    REQUIRE d.device_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT merchant_id_unique IF NOT EXISTS
    FOR (m:Merchant)
    REQUIRE m.merchant_id IS UNIQUE
    """
]

with driver.session() as session:
    for constraint in constraints:
        session.run(constraint)

print("All uniqueness constraints created successfully!")

All uniqueness constraints created successfully!


In [26]:
with driver.session() as session:
    result = session.run("SHOW CONSTRAINTS")

    for record in result:
        print(
            "Name:", record["name"],
            "| Type:", record["type"],
            "| Properties:", record["properties"]
        )

Name: customer_id_unique | Type: NODE_PROPERTY_UNIQUENESS | Properties: ['customer_id']
Name: device_id_unique | Type: NODE_PROPERTY_UNIQUENESS | Properties: ['device_id']
Name: merchant_id_unique | Type: NODE_PROPERTY_UNIQUENESS | Properties: ['merchant_id']


## Create Nodes and Relationships

In [27]:
# Prepare unique records and create Customer nodes in Neo4j using Cypher MERGE
customer_records = (customers_df[["customer_id"]].dropna().drop_duplicates().to_dict("records"))
query = """
UNWIND $records AS record
MERGE (c:Customer {customer_id: record.customer_id})
"""

with driver.session() as session:
    result = session.run(query, records=customer_records)
    result.consume()
print("Customer nodes processed:", len(customer_records))

# Prepare unique records and create Device nodes in Neo4j using Cypher MERGE
device_records = (devices_df[["device_id"]].dropna().drop_duplicates().to_dict("records"))
query = """
UNWIND $records AS record
MERGE (d:Device {device_id: record.device_id})
"""

with driver.session() as session:
    result = session.run(query, records=device_records)
    result.consume()
print("Device nodes processed:", len(device_records))

# Prepare unique records and create Merchant nodes in Neo4j using Cypher MERGE
merchant_records = (merchants_df[["merchant_id"]].dropna().drop_duplicates().to_dict("records"))
query = """
UNWIND $records AS record
MERGE (m:Merchant {merchant_id: record.merchant_id})
"""

with driver.session() as session:
    result = session.run(query, records=merchant_records)
    result.consume()
print("Merchant nodes processed:", len(merchant_records))

Customer nodes processed: 1577
Device nodes processed: 1705
Merchant nodes processed: 200


In [28]:
# Create USED_DEVICE relationships between customers and their devices, skipping transactions with missing device IDs
used_device_records = [
    {
        "customer_id": t["customer_id"],
        "device_id": t["device_id"]
    }
    for t in transactions_for_neo4j
    if t["device_id"] is not None
]

query = """
UNWIND $records AS record
MATCH (c:Customer {customer_id: record.customer_id})
MATCH (d:Device {device_id: record.device_id})
MERGE (c)-[:USED_DEVICE]->(d)
"""

with driver.session() as session:
    result = session.run(query, records=used_device_records)
    result.consume()

print("USED_DEVICE relationships processed:", len(used_device_records))

USED_DEVICE relationships processed: 5000


In [29]:
# Create PAID relationships between customers and merchants
# Storing transaction details such as amount, time and fraud label

paid_records = [
    {
        "transaction_id": t["transaction_id"],
        "customer_id": t["customer_id"],
        "merchant_id": t["merchant_id"],
        "amount": t["amount"],
        "time": t["time"],
        "fraud_label": t["fraud_label"]
    }
    for t in transactions_for_neo4j
]

query = """
UNWIND $records AS record
MATCH (c:Customer {customer_id: record.customer_id})
MATCH (m:Merchant {merchant_id: record.merchant_id})
MERGE (c)-[p:PAID {transaction_id: record.transaction_id}]->(m)
SET p.amount = record.amount,
    p.time = record.time,
    p.fraud_label = record.fraud_label
"""

with driver.session() as session:
    result = session.run(query, records=paid_records)
    result.consume()

print("PAID relationships processed:", len(paid_records))

PAID relationships processed: 5000


In [30]:
# Prepare valid device usage records by grouping transaction timestamps for each customer device pair
valid_device_usage = neo4j_df.dropna(subset=["device_id", "transaction_time_parsed"]).copy()

device_usage_df = (
    valid_device_usage
    .groupby(["customer_id", "device_id"])["transaction_time_parsed"]
    .apply(lambda timestamps: [
        timestamp.to_pydatetime()
        for timestamp in timestamps
    ])
    .reset_index(name="usage_times")
)

device_usage_records = device_usage_df.to_dict("records")
print("Unique customer-device pairs:", len(device_usage_records))
print("Sample record:")
print(device_usage_records[0])

Unique customer-device pairs: 1712
Sample record:
{'customer_id': 'C_0004ff2f7739f0d3', 'device_id': 'D_506a929cba09f1ad', 'usage_times': [datetime.datetime(2024, 10, 9, 13, 38, 16, 727000, tzinfo=datetime.timezone.utc), datetime.datetime(2024, 10, 9, 13, 39, 10, 860000, tzinfo=datetime.timezone.utc)]}


In [31]:
# Update USED_DEVICE relationships with usage timestamps for each customer device pair
query_device_usage = """
UNWIND $records AS record
MATCH (c:Customer {customer_id: record.customer_id})
MATCH (d:Device {device_id: record.device_id})
MERGE (c)-[r:USED_DEVICE]->(d)
SET r.usage_times = record.usage_times
"""

with driver.session() as session:
    result = session.run(
        query_device_usage,
        records=device_usage_records
    )
    result.consume()

In [32]:
with driver.session() as session:
    result = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS relationship_type,
               count(r) AS total
    """)

    for record in result:
        print(
            f"{record['relationship_type']}: "
            f"{record['total']}"
        )

USED_DEVICE: 1714
PAID: 5000


Observation: although 5000 transactions were processed for USED_DEVICE, Neo4j created only 1714 relationships. That's because MERGE avoids creating duplicate relationships when the same customer uses the same device repeatedly

## Fraud Detection 

In [33]:
# Identify devices used by three or more distinct customers within a 7-day period 
query_shared_devices_7days = """
MATCH (c:Customer)-[r:USED_DEVICE]->(d:Device)
UNWIND r.usage_times AS used_at

WITH d,
     collect({
         customer_id: c.customer_id,
         used_at: used_at
     }) AS events

UNWIND events AS anchor

WITH d, anchor,
     [event IN events
      WHERE event.used_at >= anchor.used_at
      AND event.used_at <= anchor.used_at + duration({days: 7})
     ] AS window_events

UNWIND window_events AS event

WITH d,
     anchor.used_at AS window_start,
     collect(DISTINCT event.customer_id) AS customers

WHERE size(customers) >= 3

RETURN
    d.device_id AS device_id,
    window_start,
    window_start + duration({days: 7}) AS window_end,
    size(customers) AS customer_count,
    customers

ORDER BY customer_count DESC, window_start
"""

with driver.session() as session:
    results = session.run(query_shared_devices_7days)
    shared_device_results = [dict(record) for record in results]

for record in shared_device_results:
    print(record)
print("\nTotal suspicious device windows:", len(shared_device_results))

{'device_id': 'D_0c947fb45bb50e72', 'window_start': neo4j.time.DateTime(2024, 10, 25, 23, 5, 32, 631000000, tzinfo=<UTC>), 'window_end': neo4j.time.DateTime(2024, 11, 1, 23, 5, 32, 631000000, tzinfo=<UTC>), 'customer_count': 3, 'customers': ['C_984cb84b0a294365', 'C_1aa451e90647f952', 'C_148927ad36a84923']}
{'device_id': 'D_0c947fb45bb50e72', 'window_start': neo4j.time.DateTime(2024, 10, 25, 23, 5, 55, 251000000, tzinfo=<UTC>), 'window_end': neo4j.time.DateTime(2024, 11, 1, 23, 5, 55, 251000000, tzinfo=<UTC>), 'customer_count': 3, 'customers': ['C_984cb84b0a294365', 'C_1aa451e90647f952', 'C_148927ad36a84923']}
{'device_id': 'D_0c947fb45bb50e72', 'window_start': neo4j.time.DateTime(2024, 10, 25, 23, 6, 35, 495000000, tzinfo=<UTC>), 'window_end': neo4j.time.DateTime(2024, 11, 1, 23, 6, 35, 495000000, tzinfo=<UTC>), 'customer_count': 3, 'customers': ['C_984cb84b0a294365', 'C_1aa451e90647f952', 'C_148927ad36a84923']}
{'device_id': 'D_0c947fb45bb50e72', 'window_start': neo4j.time.DateTime(2

This is a potentially suspicious pattern because three different customer accounts are using the same device within seven days.

In [34]:
# Find customers connected within two relationship hops to customers involved in fraudulent payments
query_fraud_customers = """
MATCH (fraud_customer:Customer)-[p:PAID]->(m:Merchant)
WHERE p.fraud_label = 1

MATCH path = (related_customer:Customer)-[*0..2]-(fraud_customer)

RETURN DISTINCT
    related_customer.customer_id AS customer_id,
    fraud_customer.customer_id AS fraud_customer_id,
    p.transaction_id AS fraud_transaction_id,
    p.amount AS fraud_amount,
    length(path) AS hops

ORDER BY hops, fraud_amount DESC
"""

with driver.session() as session:
    results = session.run(query_fraud_customers)

    fraud_customer_results = [dict(record) for record in results]
print("Total connected customer records:", len(fraud_customer_results))

Total connected customer records: 944


In [35]:
# Find the top 10 merchants receiving payments from customers who share the same device
query_shared_device_merchants = """
MATCH (c:Customer)-[:USED_DEVICE]->(d:Device)
WITH d, collect(DISTINCT c) AS customers
WHERE size(customers) >= 2

UNWIND customers AS customer

MATCH (customer)-[p:PAID]->(m:Merchant)

WITH m,
     count(DISTINCT p.transaction_id) AS shared_device_payments,
     collect(DISTINCT d.device_id) AS shared_devices

RETURN
    m.merchant_id AS merchant_id,
    shared_device_payments,
    size(shared_devices) AS shared_device_count,
    shared_devices

ORDER BY shared_device_payments DESC
LIMIT 10
"""

with driver.session() as session:
    results = session.run(query_shared_device_merchants)

    merchant_results = [dict(record) for record in results]

for record in merchant_results:
    print(record)

{'merchant_id': 'M_8094417e158fefdb', 'shared_device_payments': 9, 'shared_device_count': 1, 'shared_devices': ['D_fd8b0f414ced6d12']}
{'merchant_id': 'M_e9d157b33b203fe1', 'shared_device_payments': 9, 'shared_device_count': 1, 'shared_devices': ['D_d94ce79d00671ba9']}
{'merchant_id': 'M_ce0b45ab7127ca6a', 'shared_device_payments': 9, 'shared_device_count': 1, 'shared_devices': ['D_d94ce79d00671ba9']}
{'merchant_id': 'M_fb626fe084cf053e', 'shared_device_payments': 8, 'shared_device_count': 1, 'shared_devices': ['D_184ed5721b197852']}
{'merchant_id': 'M_93f8de771fbee5e9', 'shared_device_payments': 8, 'shared_device_count': 1, 'shared_devices': ['D_650d3838258b1e63']}
{'merchant_id': 'M_7b4b46d8ae7f95f1', 'shared_device_payments': 8, 'shared_device_count': 1, 'shared_devices': ['D_184ed5721b197852']}
{'merchant_id': 'M_06868839e93b9a80', 'shared_device_payments': 7, 'shared_device_count': 1, 'shared_devices': ['D_0c947fb45bb50e72']}
{'merchant_id': 'M_d945de86dbb6eb24', 'shared_device_pa

This analysis helps SENTINEL identify merchants connected to shared device payment activity. These are potential investigation signals.

Run this following Cypher query in Neo4j Aura

This will display:
- One shared Device node.
- Three connected Customer nodes.
- Their payment relationships.
- The corresponding Merchant nodes.

Observation : The graph visualisation identifies a device shared by three distinct customers within a seven-day period. Each customer is connected to the same device through a USED_DEVICE relationship and has payment relationships with merchants through PAID edges. Such shared device activity may indicate coordinated account usage or potentially suspicious payment behaviour. 

In [ ]:
MATCH (c:Customer)-[u:USED_DEVICE]->(d:Device {
    device_id: "D_0c947fb45bb50e72"
})
OPTIONAL MATCH (c)-[p:PAID]->(m:Merchant)

RETURN c, u, d, p, m
LIMIT 50

## Final Summary

In [36]:
with driver.session() as session:

    result = session.run("""
        MATCH (n)
        RETURN labels(n) AS node_type, count(n) AS total
    """)

    print("NODE COUNTS")
    for record in result:
        print(record["node_type"], ":", record["total"])

    result = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS relationship_type, count(r) AS total
    """)

    print("\nRELATIONSHIP COUNTS")
    for record in result:
        print(record["relationship_type"], ":", record["total"])

    result = session.run("""
        MATCH ()-[p:PAID]->()
        WHERE p.fraud_label = 1
        RETURN count(p) AS fraudulent_payments
    """)

    print("\nFRAUDULENT PAYMENTS:", 
          result.single()["fraudulent_payments"])

NODE COUNTS
['Customer'] : 1577
['Device'] : 1705
['Merchant'] : 200

RELATIONSHIP COUNTS
USED_DEVICE : 1714
PAID : 5000

FRAUDULENT PAYMENTS: 25
